In [21]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt

from train.prep import NBADataset

In [20]:
try:
    assert torch.cuda.is_available()
    device = torch.device("cuda")
except:
    device = torch.device("cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
criterion = torch.nn.CrossEntropyLoss()
def train_loop(net, ds, n_epochs):
    train_set, val_set, test_set = torch.utils.data.random_split(ds, [0.8, 0.1, 0.1])
    train_dl, val_dl, test_dl = DataLoader(train_set, batch_size=128, shuffle=True), DataLoader(val_set, batch_size=128), DataLoader(test_set, batch_size=128)
    best_val_loss = None
    val_losses = []
    optimizer = torch.optim.Adam(net.parameters(), lr=5e-3)
    for epoch in range(n_epochs):
        if (epoch+1)%10==0:
            print('Epoch', epoch+1)
        net.train()
        for batch, labels in train_dl:
            batch = batch.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            out = net.forward(batch)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            del batch, labels, out, loss

        net.eval()
        val_loss = 0
        for batch, labels in val_dl:
            batch = batch.to(device)
            labels = labels.to(device)
            out = net.forward(batch)
            val_loss += criterion(out, labels).item()
            del batch, labels, out
        val_losses.append(val_loss)
        if best_val_loss is None or val_loss < best_val_loss:
            torch.save(net.state_dict(), './nba_predictor.pt')
            best_val_loss = val_loss
        torch.cuda.empty_cache()

    plt.plot([i for i in range(len(val_losses))], val_losses)

    net.load_state_dict(torch.load('./nba_predictor.pt', weights_only=True))

    test_correct = 0
    total = len(test_set)
    for batch, labels in test_dl:
        batch = batch.to(device)
        labels = labels.to(device)
        out = net.forward(batch)
        test_correct += (torch.argmax(out, dim=1) == labels).sum()
        del batch, labels, out
        torch.cuda.empty_cache()
    print(f"Test accuracy: {test_correct/total}")